# LeRobot Dataset Inspector
Explore episodes exported by `convert_to_lerobot.py`. No Hugging Face dependencies needed — pure pandas + pyarrow + matplotlib.

In [ ]:
import json
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pyarrow.parquet as pq

DATASET_DIR = pathlib.Path("./lerobot_dataset")

## Dataset summary

In [ ]:
info_path = DATASET_DIR / "meta" / "info.json"
with open(info_path) as f:
    info = json.load(f)

print(f"Robot type      : {info['robot_type']}")
print(f"FPS             : {info['fps']}")
print(f"Total episodes  : {info['total_episodes']}")
print(f"Total frames    : {info['total_frames']}")
print(f"Total videos    : {info['total_videos']}")
print(f"LeRobot version : {info['codebase_version']}")
print()
print("Tasks:")
for t in info["tasks"]:
    print(f"  [{t['task_index']}] {t['task'] or '(no instruction)'}")
print()
print("Features:")
for name, feat in info["features"].items():
    print(f"  {name:40s}  dtype={feat['dtype']:8s}  shape={feat['shape']}")

## List all episode parquet files

In [ ]:
parquet_files = sorted((DATASET_DIR / "data").glob("**/*.parquet"))
print(f"{len(parquet_files)} parquet file(s) found")
for p in parquet_files:
    tbl = pq.read_table(p)
    print(f"  {p.name:35s}  rows={len(tbl)}")

## Load a single episode

In [ ]:
EPISODE_IDX = 0   # <-- change this to inspect a different episode

chunk = f"chunk-{EPISODE_IDX // 1000:03d}"
ep_file = DATASET_DIR / "data" / chunk / f"episode_{EPISODE_IDX:06d}.parquet"

df = pd.read_parquet(ep_file)
print(f"Episode {EPISODE_IDX}: {len(df)} frames  "
      f"duration={df['timestamp'].iloc[-1]:.2f}s")
df.head()

## Joint positions: commanded vs actual

In [ ]:
t = df["timestamp"].to_numpy()

cmd    = np.stack(df["action"].to_numpy())[:, :6]      # first 6 dims are joints
actual = np.stack(df["observation.state"].to_numpy())

fig, axes = plt.subplots(3, 2, figsize=(13, 9), sharex=True)
axes = axes.flatten()
for j in range(6):
    axes[j].plot(t, cmd[:, j],    label="cmd",    lw=1.2)
    axes[j].plot(t, actual[:, j], label="actual", lw=1.2, linestyle="--")
    axes[j].set_title(f"J{j+1}")
    axes[j].set_ylabel("deg")
    axes[j].legend(fontsize=8)
    axes[j].grid(True, alpha=0.3)

axes[-1].set_xlabel("time (s)")
axes[-2].set_xlabel("time (s)")
fig.suptitle(f"Episode {EPISODE_IDX} — Joint positions", fontsize=12)
plt.tight_layout()
plt.show()

## Gripper state

In [ ]:
gripper = np.stack(df["action"].to_numpy())[:, 6]   # 7th action dim

plt.figure(figsize=(10, 3))
plt.plot(t, gripper, lw=1.5, color="steelblue")
plt.axhline(0.65, color="green",  ls="--", lw=0.9, label="open threshold (0.65)")
plt.axhline(0.35, color="orange", ls="--", lw=0.9, label="close threshold (0.35)")
plt.ylim(-0.05, 1.05)
plt.xlabel("time (s)")
plt.ylabel("gripper_norm")
plt.title(f"Episode {EPISODE_IDX} — Gripper")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## EEF trajectory (XYZ)

In [ ]:
eef = np.stack(df["observation.eef_pose"].to_numpy())
labels = ["x_mm", "y_mm", "z_mm", "rx_deg", "ry_deg", "rz_deg"]

fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
axes = axes.flatten()
for i, lbl in enumerate(labels):
    axes[i].plot(t, eef[:, i], lw=1.2)
    axes[i].set_title(lbl)
    axes[i].grid(True, alpha=0.3)
    if i >= 3:
        axes[i].set_xlabel("time (s)")

fig.suptitle(f"Episode {EPISODE_IDX} — EEF pose", fontsize=12)
plt.tight_layout()
plt.show()

## 3-D EEF path

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(7, 6))
ax  = fig.add_subplot(111, projection="3d")
sc  = ax.scatter(eef[:, 0], eef[:, 1], eef[:, 2],
                 c=t, cmap="viridis", s=4)
plt.colorbar(sc, ax=ax, label="time (s)")
ax.set_xlabel("X (mm)"); ax.set_ylabel("Y (mm)"); ax.set_zlabel("Z (mm)")
ax.set_title(f"Episode {EPISODE_IDX} — EEF 3-D path")
plt.tight_layout()
plt.show()

## Dataset-wide statistics

In [ ]:
durations = []
frame_counts = []
task_counts: dict[str, int] = {}
task_names = {t["task_index"]: t["task"] for t in info["tasks"]}

for p in parquet_files:
    d = pd.read_parquet(p)
    durations.append(float(d["timestamp"].iloc[-1]))
    frame_counts.append(len(d))
    tidx = int(d["task_index"].iloc[0])
    task_label = task_names.get(tidx, str(tidx))
    task_counts[task_label] = task_counts.get(task_label, 0) + 1

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(durations, bins=20, color="steelblue", edgecolor="white")
axes[0].set_xlabel("duration (s)"); axes[0].set_title("Episode durations")

axes[1].hist(frame_counts, bins=20, color="salmon", edgecolor="white")
axes[1].set_xlabel("frames"); axes[1].set_title("Frames per episode")

axes[2].bar(range(len(task_counts)), list(task_counts.values()), color="mediumpurple", edgecolor="white")
axes[2].set_xticks(range(len(task_counts)))
short_labels = [lbl[:30] + "..." if len(lbl) > 30 else lbl for lbl in task_counts]
axes[2].set_xticklabels(short_labels, rotation=20, ha="right", fontsize=8)
axes[2].set_title("Episodes per task")

plt.tight_layout()
plt.show()

print(f"Mean duration : {np.mean(durations):.2f}s  (min {np.min(durations):.2f}s, max {np.max(durations):.2f}s)")
print(f"Mean frames   : {np.mean(frame_counts):.0f}")

## Play video (if available)

In [ ]:
import cv2
from IPython.display import Image as IPImage, display
import io

chunk = f"chunk-{EPISODE_IDX // 1000:03d}"
vid_path = (DATASET_DIR / "videos" / "observation.images.wrist_cam"
            / chunk / f"episode_{EPISODE_IDX:06d}.mp4")

if not vid_path.exists():
    print(f"No video for episode {EPISODE_IDX} — camera was not attached during recording.")
else:
    # Show first frame
    cap = cv2.VideoCapture(str(vid_path))
    ok, frame = cap.read()
    cap.release()
    if ok:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        _, buf = cv2.imencode(".jpg", frame)
        display(IPImage(data=buf.tobytes()))
        print(f"Video: {vid_path.name}  (showing frame 0)")
    else:
        print("Could not read first frame from video.")

## Raw parquet peek

In [ ]:
# Expand list columns into individual scalar columns for easy inspection
def expand_df(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["timestamp"]     = df["timestamp"]
    out["frame_index"]   = df["frame_index"]
    out["episode_index"] = df["episode_index"]
    out["task_index"]    = df["task_index"]
    out["next.done"]     = df["next.done"]
    state_arr = np.stack(df["observation.state"].to_numpy())
    for j in range(6):
        out[f"actual_j{j+1}"] = state_arr[:, j]
    act_arr = np.stack(df["action"].to_numpy())
    for j in range(6):
        out[f"cmd_j{j+1}"] = act_arr[:, j]
    out["gripper"] = act_arr[:, 6]
    return out

expand_df(df)